# Async/Await/Corutines — Multilenguaje (Python, C#, Java, Rust, C++)

**Objetivo:** practicar *concurrencia cooperativa* (no paralelismo) en distintos lenguajes.
- **No confundas:** `async/await` coordina tareas; **no** garantiza ejecución en paralelo.
- El **paralelismo real** depende del SO, del *runtime* y de los **núcleos disponibles**.
- Las demos usan *sleep/retardo* como simulación de I/O.

> En Colab, primero corre la celda de **instalación** (tarda algunos minutos).


## 0) Instalación de herramientas (una sola vez)

In [4]:
%%bash
set -e
apt-get update -qq
# C++ y Java
apt-get install -y -qq g++ openjdk-17-jdk
# C# (Mono)
apt-get install -y -qq mono-devel
# Rust (rustup + cargo)
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
source $HOME/.cargo/env
rustc --version || true
g++ --version | head -n1
javac -version
mcs --version | head -n1
echo 'Listo.'


Selecting previously unselected package fonts-dejavu-core.
(Reading database ... 126435 files and directories currently installed.)
Preparing to unpack .../00-fonts-dejavu-core_2.37-2build1_all.deb ...
Unpacking fonts-dejavu-core (2.37-2build1) ...
Selecting previously unselected package fonts-dejavu-extra.
Preparing to unpack .../01-fonts-dejavu-extra_2.37-2build1_all.deb ...
Unpacking fonts-dejavu-extra (2.37-2build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package libxxf86dga1:amd64.
Preparing to unpack .../03-libxxf86dga1_2%3a1.1.5-0ubuntu3_amd64.deb ...
Unpacking libxxf86dga1:amd64 (2:1.1.5-0ubuntu3) ...
Selecting previously unselected package x11-utils.
Preparing to unpack .../04-x11-utils_7.7+5build2_amd64.deb ...
Unpacking x11-utils (7.7+5build2) ...
Selecting previously unselected package libatk-wrapper-java.
Pre

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
info: downloading installer
info: profile set to 'default'
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for 'stable-x86_64-unknown-linux-gnu'
info: latest update on 2025-08-07, rust version 1.89.0 (29483883e 2025-08-04)
info: downloading component 'cargo'
info: downloading component 'clippy'
info: downloading component 'rust-docs'
info: downloading component 'rust-std'
info: downloading component 'rustc'
info: downloading component 'rustfmt'
info: installing component 'cargo'
info: installing component 'clippy'
info: installing component 'rust-docs'
info: installing component 'rust-std'
info: installing component 'rustc'
info: installing component 'rustfmt'
info: default toolchain set to 'stable-x86_64-unknown-linux-gnu'


## 1) Python — `asyncio` (concurrencia I/O)
**Idea:** lanzar N tareas que hacen `await asyncio.sleep()` (simula I/O).

In [5]:
import asyncio, time
N=50; SLEEP=0.1

def io_blocking():
    time.sleep(SLEEP); return 1

print("Secuencial I/O...")
t0=time.perf_counter()
_ = sum(io_blocking() for _ in range(N))
print("Tiempo secuencial:", time.perf_counter()-t0, "seg")

async def tarea(i):
    await asyncio.sleep(SLEEP); return i

async def main():
    t0=time.perf_counter()
    res = await asyncio.gather(*(tarea(i) for i in range(N)))
    print("Async I/O:", time.perf_counter()-t0, "seg", "| tareas:", len(res))

await main()  # En notebook: usar await (no asyncio.run)


Secuencial I/O...
Tiempo secuencial: 5.007267887000012 seg
Async I/O: 0.1016857429999618 seg | tareas: 50


## 2) C# — `async`/`await` con `Task`
**Nota:** compilamos con Mono. Es concurrencia; el paralelismo no está garantizado.


In [9]:
%%bash
cat > Program.cs << 'CS'
using System;
using System.Threading.Tasks;
using System.Diagnostics;

class P
{
  static async Task<int> F(int x)
  {
    await Task.Delay(100); // simula I/O
    return x*x;
  }

  // Método async real
  static async Task<int[]> RunAsync()
  {
    var sw = Stopwatch.StartNew();
    var t1 = F(5); var t2 = F(7); var t3 = F(9);
    var r = await Task.WhenAll(t1, t2, t3);
    sw.Stop();
    Console.WriteLine("Resultados: " + string.Join(",", r));
    Console.WriteLine("Tiempo (async): " + sw.ElapsedMilliseconds + " ms");
    return r;
  }

  // Punto de entrada compatible con mcs/Mono:
  public static void Main(string[] args)
  {
    // Bloquea hasta que termine la async:
    RunAsync().GetAwaiter().GetResult();
  }
}
CS

In [10]:
!ls -la

total 20
drwxr-xr-x 1 root root 4096 Sep 17 14:00 .
drwxr-xr-x 1 root root 4096 Sep 17 13:58 ..
drwxr-xr-x 4 root root 4096 Sep 15 17:50 .config
-rw-r--r-- 1 root root  729 Sep 17 14:06 Program.cs
drwxr-xr-x 1 root root 4096 Sep 15 17:50 sample_data


In [11]:
!mcs -langversion:latest Program.cs -out:prog.exe
!time mono prog.exe


Resultados: 25,49,81
Tiempo (async): 107 ms

real	0m0.147s
user	0m0.045s
sys	0m0.010s


## 3) Java — `CompletableFuture`
**Composición** de tareas asíncronas. Concurrencia para I/O.


In [13]:
%%bash
cat > Demo.java << 'JAVA'
import java.util.concurrent.*;
public class Demo
{
  static void sleep(long ms){ try{ Thread.sleep(ms);}catch(Exception e){} }
  public static void main(String[] a) throws Exception
  {
    long t0 = System.currentTimeMillis();
    var f1 = CompletableFuture.supplyAsync(() -> { sleep(100); return 25; });
    var f2 = CompletableFuture.supplyAsync(() -> { sleep(100); return 49; });
    var f3 = CompletableFuture.supplyAsync(() -> { sleep(100); return 81; });
    var all = CompletableFuture.allOf(f1,f2,f3);
    all.join();
    int sum = f1.get() + f2.get() + f3.get();
    System.out.println("Suma: " + sum);
    System.out.println("Tiempo (async): " + (System.currentTimeMillis()-t0) + " ms");
  }
}
JAVA


In [12]:
!ls -la

total 28
drwxr-xr-x 1 root root 4096 Sep 17 14:06 .
drwxr-xr-x 1 root root 4096 Sep 17 13:58 ..
drwxr-xr-x 4 root root 4096 Sep 15 17:50 .config
-rwxr-xr-x 1 root root 5632 Sep 17 14:06 prog.exe
-rw-r--r-- 1 root root  729 Sep 17 14:06 Program.cs
drwxr-xr-x 1 root root 4096 Sep 15 17:50 sample_data


In [14]:
!javac Demo.java
!time java Demo

Suma: 155
Tiempo (async): 130 ms

real	0m0.187s
user	0m0.090s
sys	0m0.027s


## 4) Rust — Tokio (runtime async)
Necesitamos **cargo** para dependencias. Esto crea un mini proyecto y lo corre.


In [22]:
%%bash
set -e
source $HOME/.cargo/env
# Crear proyecto
cargo new --bin async_demo >/dev/null 2>&1 || true
cd async_demo
# Cargo.toml con tokio
cat > Cargo.toml << 'TOML'
[package]
name = "async_demo"
version = "0.1.0"
edition = "2021"

[dependencies]
tokio = { version = "1", features = ["full"] }
TOML
# main.rs
cat > src/main.rs << 'RS'
use tokio::time::{sleep, Duration};
use std::time::Instant;

#[tokio::main]
async fn main()
{
    let t0 = Instant::now();
    let t1 = tokio::spawn(async { sleep(Duration::from_millis(100)).await; 25 });
    let t2 = tokio::spawn(async { sleep(Duration::from_millis(100)).await; 49 });
    let t3 = tokio::spawn(async { sleep(Duration::from_millis(100)).await; 81 });
    let (a,b) = tokio::join!(t1, t2);
    let a = a.unwrap(); let b = b.unwrap();
    let c = t3.await.unwrap();
    println!("Suma: {}", a + b + c);
    println!("Tiempo (async): {} ms", t0.elapsed().as_millis());
}
RS
time cargo run --quiet


Suma: 155
Tiempo (async): 101 ms



real	0m24.803s
user	0m27.541s
sys	0m4.012s


In [23]:
!ls -la; ls -la ./async_demo/

total 40
drwxr-xr-x 1 root root 4096 Sep 17 14:13 .
drwxr-xr-x 1 root root 4096 Sep 17 13:58 ..
drwxr-xr-x 5 root root 4096 Sep 17 14:16 async_demo
drwxr-xr-x 4 root root 4096 Sep 15 17:50 .config
-rw-r--r-- 1 root root 2304 Sep 17 14:08 Demo.class
-rw-r--r-- 1 root root  706 Sep 17 14:08 Demo.java
-rwxr-xr-x 1 root root 5632 Sep 17 14:06 prog.exe
-rw-r--r-- 1 root root  729 Sep 17 14:06 Program.cs
drwxr-xr-x 1 root root 4096 Sep 15 17:50 sample_data
total 40
drwxr-xr-x 5 root root 4096 Sep 17 14:16 .
drwxr-xr-x 1 root root 4096 Sep 17 14:13 ..
-rw-r--r-- 1 root root 9658 Sep 17 14:16 Cargo.lock
-rw-r--r-- 1 root root  128 Sep 17 14:16 Cargo.toml
drwxr-xr-x 6 root root 4096 Sep 17 14:13 .git
-rw-r--r-- 1 root root    8 Sep 17 14:13 .gitignore
drwxr-xr-x 2 root root 4096 Sep 17 14:13 src
drwxr-xr-x 3 root root 4096 Sep 17 14:16 target


## 5) C++20 — Coroutines (demo conceptual)
Usamos `co_await` con un *awaiter* que simula I/O con `sleep_for`.


**Comentarios:** C++20 introdujo corrutinas con las palabras clave co_await, co_yield y co_return. A diferencia de Python/C#/Rust, C++ no provee un scheduler por defecto: las corrutinas son de bajo nivel y requieren definir cómo se suspenden/retoman (mediante tipos de promesa, awaiter, etc.). Librerías o frameworks pueden construir sobre esto (por ejemplo, futuros de std::async, o librerías como Boost.Asio, cppcoro o mecanismos en C++23) para lograr comportamiento tipo async/await. La sintaxis hace que el código asíncrono parezca secuencial, pero la complejidad de implementación es alta. Conclusión: C++ ofrece gran control y rendimiento en concurrencia, a costa de una curva de aprendizaje más pronunciada y mayor responsabilidad del programador para la sincronización y manejo de memoria.

In [26]:
%%bash
cat > coro.cpp << 'CPP'
#include <iostream>
#include <coroutine>
#include <thread>
#include <chrono>
using namespace std::chrono_literals;

// Awaiter que reanuda la corrutina desde otro hilo después de dormir
struct SleepAwaiter
{
  std::chrono::milliseconds dur;
  bool await_ready() const noexcept { return false; }
  void await_suspend(std::coroutine_handle<> h) const
  {
    std::thread([h,d=*this]()
    {
      std::this_thread::sleep_for(d.dur);
      h.resume();                   // <- reanudar aquí
    }).detach();
  }
  void await_resume() const noexcept {}
};

// Tarea mínima que guarda el handle para poder "esperar"
struct SimpleTask
{
  struct promise_type
  {
    SimpleTask get_return_object()
    {
      return SimpleTask{ std::coroutine_handle<promise_type>::from_promise(*this) };
    }
    std::suspend_never initial_suspend() noexcept { return {}; }
    std::suspend_always final_suspend() noexcept { return {}; } // deja el frame hasta que lo destruyamos
    void return_void() noexcept {}
    void unhandled_exception() { std::terminate(); }
  };

  std::coroutine_handle<promise_type> h;
  explicit SimpleTask(std::coroutine_handle<promise_type> h): h(h) {}
  SimpleTask(SimpleTask&& o) noexcept : h(std::exchange(o.h, {})) {}
  ~SimpleTask(){ if (h) h.destroy(); }

  // Bloquear hasta que termine (espera activa simple para demo)
  void wait() const
  {
    while (!h.done()) std::this_thread::sleep_for(1ms);
  }
};

SimpleTask demo()
{
  std::cout << "inicio\n";
  co_await SleepAwaiter{100ms};    // simula I/O y reanuda luego
  std::cout << "fin\n";
}

int main()
{
  auto t = demo();  // comienza, imprime "inicio" y se suspende
  t.wait();         // <- aquí esperamos a que se reanude y termine
}
CPP

g++ -std=gnu++20 coro.cpp -O2 -pthread -o coro
time ./coro


inicio
fin



real	0m0.104s
user	0m0.003s
sys	0m0.001s


### Conclusión
- `async/await` es **concurrencia**. El paralelismo **no está garantizado**.
- Los *runtimes* (Tokio, .NET, JVM) y el SO deciden cómo despachar las tareas.
- Para **CPU-bound**, preferí *thread pools* o *processes* (según el lenguaje).
